# Data‑Cleaning Techniques Comparison Notebook

This notebook demonstrates nine common data‑cleaning approaches applied **on copies** of the original dataset `AI_Powered_IoT_Network_Intrusion_Detection_Dataset.csv`.  The original CSV is never altered; each technique writes its result to a separate CSV file prefixed with the technique name.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Path to the original dataset (do not modify)
DATA_PATH = Path(r'D:/work/dmProj/AI_Powered_IoT_Network_Intrusion_Detection_Dataset.csv')

# Helper to save a cleaned copy
def save_copy(df: pd.DataFrame, suffix: str):
    out_path = DATA_PATH.with_name(f'{DATA_PATH.stem}_{suffix}{DATA_PATH.suffix}')
    df.to_csv(out_path, index=False)
    print(f'Saved {suffix} version to {out_path}')


## 1️⃣ Mean Imputation

In [ ]:
# Load original data
df = pd.read_csv(DATA_PATH)
# Create a copy for this technique
df_mean = df.copy()
# Mean imputation for numeric columns
for col in df_mean.select_dtypes(include='number').columns:
    df_mean[col].fillna(df_mean[col].mean(), inplace=True)
# Save the result
save_copy(df_mean, 'mean_imputed')


## 2️⃣ Median Imputation

In [ ]:
df_median = df.copy()
for col in df_median.select_dtypes(include='number').columns:
    df_median[col].fillna(df_median[col].median(), inplace=True)
save_copy(df_median, 'median_imputed')


## 3️⃣ Mode Imputation

In [ ]:
df_mode = df.copy()
for col in df_mode.columns:
    mode_val = df_mode[col].mode()[0]
    df_mode[col].fillna(mode_val, inplace=True)
save_copy(df_mode, 'mode_imputed')


## 4️⃣ Deletion (drop rows with any missing value)

In [ ]:
df_del = df.dropna()
save_copy(df_del, 'deletion')


## 5️⃣ Inter‑Quartile‑Range (IQR) Outlier Removal

In [ ]:
df_iqr = df.copy()
num = df_iqr.select_dtypes(include='number')
Q1 = num.quantile(0.25)
Q3 = num.quantile(0.75)
IQR = Q3 - Q1
mask = ~((num < (Q1 - 1.5 * IQR)) | (num > (Q3 + 1.5 * IQR))).any(axis=1)
df_iqr = df_iqr[mask]
save_copy(df_iqr, 'iqr')


## 6️⃣ Z‑Score Outlier Removal

In [ ]:
df_z = df.copy()
num = df_z.select_dtypes(include='number')
z = (num - num.mean()) / num.std()
mask = (z.abs() <= 3).all(axis=1)
df_z = df_z[mask]
save_copy(df_z, 'zscore')


## 7️⃣ Binning (`packet_count` into 5 bins)

In [ ]:
df_bin = df.copy()
# Adjust the column name if yours differs
if 'packet_count' in df_bin.columns:
    df_bin['packet_count_bin'] = pd.cut(df_bin['packet_count'], bins=5, labels=False)
save_copy(df_bin, 'binned')


## 8️⃣ Regression Smoothing (Iterative Imputer) – useful when missing values exist

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
df_reg = df.copy()
num = df_reg.select_dtypes(include='number')
imp = IterativeImputer(random_state=0, max_iter=10)
df_reg[num.columns] = imp.fit_transform(num)
save_copy(df_reg, 'reg_smoothing')


## 9️⃣ Duplicate Detection (remove exact duplicate rows)

In [ ]:
df_nodup = df.drop_duplicates()
save_copy(df_nodup, 'dedup')


---
All cleaned versions are saved beside the original CSV with suffixes such as `_mean_imputed.csv`, `_iqr.csv`, etc. You can now load any of these files for downstream modeling or analysis.